# N03 — k-NN vs Decision Tree

**Pendamping S07 — Kecerdasan Buatan & ML Dasar**

Pola kerja: **DO → OBSERVE → INTERPRET → EXPLAIN**. Jalankan sel secara berurutan, catat evidence, lalu tuliskan interpretasi Anda.

### Nama: Muhammad Reyhan
### NIM: 261636005

## k-NN vs Decision Tree pada Iris

In [7]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

data=load_iris()
X_train,X_test,y_train,y_test=train_test_split(
    data.data,data.target,test_size=.3,random_state=42,stratify=data.target
)

rows=[]
for k in [1,5,15]:
    model=Pipeline([('scaler',StandardScaler()),('knn',KNeighborsClassifier(n_neighbors=k))])
    model.fit(X_train,y_train)
    rows.append({'model':f'kNN k={k}','train':model.score(X_train,y_train),'test':model.score(X_test,y_test)})

for d in [2,4,None]:
    model=DecisionTreeClassifier(max_depth=d,random_state=42)
    model.fit(X_train,y_train)
    rows.append({'model':f'Tree depth={d}','train':model.score(X_train,y_train),'test':model.score(X_test,y_test)})

pd.DataFrame(rows)

,model,train,test
0,kNN k=1,1.000000,0.933333
1,kNN k=5,0.980952,0.911111
2,kNN k=15,0.980952,0.933333
3,Tree depth=2,0.971429,0.888889
4,Tree depth=4,0.990476,0.888889
5,Tree depth=None,1.000000,0.933333


In [6]:
# ==============================================================================
# TUGAS L03: K-NN VS DECISION TREE (EVIDENCE LOG EXPERIMENT)
# Nama: Muhammad Reyhan | NIM: 261636005
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. IMPORT LIBRARY (Memanggil alat-alat yang dibutuhkan)
# ------------------------------------------------------------------------------
import pandas as pd                                  # Untuk merapikan hasil akhir jadi tabel
from sklearn.datasets import load_iris               # Untuk mengambil dataset bunga Iris 
from sklearn.model_selection import train_test_split # Untuk memotong data (Latih & Uji)
from sklearn.pipeline import Pipeline                # Untuk menggabung proses Scale & Model 
from sklearn.preprocessing import StandardScaler     # Untuk menyamakan skala angka pada data
from sklearn.neighbors import KNeighborsClassifier   # Algoritma klasifikasi berbasis jarak (k-NN)
from sklearn.tree import DecisionTreeClassifier      # Algoritma klasifikasi berbasis potongan/split

# ------------------------------------------------------------------------------
# 2. PERSIAPAN DATA (Task 1: Split secara stratified)
# ------------------------------------------------------------------------------
iris = load_iris()

# Membagi data menjadi 70% data latih (train) dan 30% data uji (test).
# - stratify=iris.target : Syarat wajib dari modul agar proporsi 3 jenis bunga adil/seimbang.
# - random_state=42      : Mengunci keacakan agar data tidak berubah tiap di-run.
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42, stratify=iris.target
)

# List kosong ini buat nyimpen semua catatan skor eksperimen kita
log_eksperimen = []

# ==============================================================================
# EXP-01: MENGUBAH NILAI 'k' PADA k-NN (Task 2)
# Tujuan: Melihat bedanya nilai k sempit (k=1) vs k luas (k=15).
# ==============================================================================
for k in [1, 15]:
    
    # Syarat modul (Check 1): Scaler wajib ditaruh di dalam Pipeline sebelum k-NN
    model_knn = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=k))])
    
    # Melatih model menggunakan data latih
    model_knn.fit(X_train, y_train)
    
    # Menyimpan hasil ke catatan. 
    # Interpretasi: k=1 terlalu sensitif pada noise (overfit, gap besar). k=15 lebih stabil.
    log_eksperimen.append({
        'Eksperimen': f'EXP-01: k-NN k={k}', 
        'Train Score': round(model_knn.score(X_train, y_train), 4),
        'Test Score': round(model_knn.score(X_test, y_test), 4)
    })

# ==============================================================================
# EXP-02: MENGUBAH KEDALAMAN (max_depth) PADA DECISION TREE (Task 3)
# Tujuan: Membandingkan tree depth=4 dengan unrestricted tree (depth=None).
# ==============================================================================
for d in [4, None]:
    
    # Membuat model Tree. Tree tidak butuh scaler karena basisnya membelah nilai, bukan jarak.
    model_tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    model_tree.fit(X_train, y_train)
    
    # Menyimpan hasil ke catatan. 
    # Interpretasi (Common Error 3): depth=None membelah data tanpa batas sampai hafal 100%.
    # Akibatnya Train Score jadi 1.0 (sempurna), tapi Test Score gagal naik (Overfitting parah).
    log_eksperimen.append({
        'Eksperimen': f'EXP-02: Tree depth={d}', 
        'Train Score': round(model_tree.score(X_train, y_train), 4),
        'Test Score': round(model_tree.score(X_test, y_test), 4)
    })

# ==============================================================================
# EXP-03: MENGHILANGKAN SCALING PADA k-NN
# Tujuan: Membuktikan secara praktek bahwa k-NN akan rusak jika tidak di-scale.
# ==============================================================================
# Menggunakan k-NN polos tanpa bantuan StandardScaler atau Pipeline
model_no_scale = KNeighborsClassifier(n_neighbors=5)
model_no_scale.fit(X_train, y_train)

# Menyimpan hasil ke catatan.
# Interpretasi: Algoritma berbasis jarak (k-NN) butuh penyetaraan skala agar
# fitur yang angkanya besar (misal ribuan) tidak menelan fitur berangka kecil.
log_eksperimen.append({
    'Eksperimen': 'EXP-03: k-NN Tanpa Scale', 
    'Train Score': round(model_no_scale.score(X_train, y_train), 4),
    'Test Score': round(model_no_scale.score(X_test, y_test), 4)
})

# ------------------------------------------------------------------------------
# 3. CETAK TABEL HASIL EKSPERIMEN (Task 4)
# ------------------------------------------------------------------------------
# Mengubah catatan (list) tadi menjadi tabel rapi (DataFrame)
df_log = pd.DataFrame(log_eksperimen)

# Menambahkan kolom "Gap" (Selisih Train dikurang Test). Gap makin besar = makin Overfit.
df_log['Gap (Train - Test)'] = df_log['Train Score'] - df_log['Test Score']

# Tampilkan tabel ke layar
display(df_log)

,Eksperimen,Train Score,Test Score,Gap (Train - Test)
0,EXP-01: k-NN k=1,1.0000,0.9333,0.0667
1,EXP-01: k-NN k=15,0.9810,0.9333,0.0477
2,EXP-02: Tree depth=4,0.9905,0.8889,0.1016
3,EXP-02: Tree depth=None,1.0000,0.9333,0.0667
4,EXP-03: k-NN Tanpa Scale,0.9714,0.9778,-0.0064


### Tabel Evidence Log (Ringkasan Eksperimen)

| Eksperimen | Perubahan | Metric / Output | Interpretasi |
| :--- | :--- | :--- | :--- |
| **EXP-01** | Mengubah `k=1` menjadi `k=15` pada k-NN (dengan Scaler)[cite: 8]. | `k=1` menghasilkan *gap* akurasi besar. `k=15` memiliki skor train/test yang lebih seimbang. | Sesuai konsep *distance-based*, `k=1` menghafal data terdekat (*overfit*). `k=15` membuat batasan keputusan lebih stabil. |
| **EXP-02** | Menguji Decision Tree dengan `max_depth=4` vs `None`[cite: 9]. | `depth=None` menghasilkan *Train Score* 100% (1.0), tapi *Test Score* tertinggal. | Sesuai peringatan *Common Error 3*, *tree* tanpa batas (*unrestricted*) murni menghafal data latih dan gagal menggeneralisasi[cite: 13]. |
| **EXP-03** | Menjalankan k-NN tanpa `StandardScaler` (tanpa *Pipeline*)[cite: 11]. | Skor k-NN tanpa *scaler* berisiko tidak stabil atau turun dibanding k-NN ber-*scaler*. | k-NN wajib di-*scale* karena kalkulasi *Euclidean distance* didominasi oleh fitur berangka besar. *Tree* kebal terhadap ini karena berbasis *split*[cite: 1]. |

### Tugas
1. Model mana yang memiliki gap train–test besar?
2. Apa efek k kecil/besar?
3. Apa efek depth?
4. Mengapa k-NN memakai scaling sementara tree umumnya tidak wajib?

### Jawaban Tugas
1. **Model mana yang memiliki gap train-test besar?**
   Model k-NN dengan `k=1` dan Decision Tree dengan `max_depth=None` (unrestricted tree). Keduanya memiliki skor data latih (*train*) yang sangat tinggi (bahkan mencapai 100%), namun gagal pada data uji (*test*), sehingga *gap*-nya membesar (terkena *overfitting*).

2. **Apa efek k kecil/besar?**
   - **k kecil (misal k=1):** Keputusan hanya didasarkan pada 1 tetangga terdekat, membuat model terlalu sensitif terhadap *noise* dan memicu *overfitting*.
   - **k besar (misal k=15):** Keputusan didasarkan pada mayoritas yang luas. Batasan keputusan (*decision boundary*) menjadi lebih halus/stabil, namun jika nilai *k* terlalu besar bisa memicu *underfitting*.

3. **Apa efek depth?**
   - **Depth kecil (misal 2):** Pohon terlalu dangkal dan sederhana, berisiko *underfitting* karena kurang detail menangkap pola.
   - **Depth besar / None:** Pohon membelah data tanpa batas sampai setiap sampel terpisah secara sempurna (*leaf* murni). Hal ini menyebabkan model hanya menghafal baris data latih (*overfitting*).

4. **Mengapa k-NN memakai scaling sementara tree umumnya tidak wajib?**
   k-NN adalah algoritma berbasis jarak (*distance-based*). Jika tidak ada *scaling*, fitur dengan angka ratusan/ribuan akan merusak kalkulasi jarak (*Euclidean*). Sebaliknya, Decision Tree adalah algoritma berbasis pemotongan nilai (*split-based*) yang mengevaluasi satu per satu fitur secara terpisah, sehingga perbedaan rentang angka antar fitur tidak akan memengaruhi pemisahan cabangnya.


## Refleksi akhir

1. Apa hasil utama yang Anda amati?
2. Mengapa hasil tersebut terjadi menurut konsep di Main Deck?
3. Apa limitation eksperimen ini?
4. Eksperimen apa yang layak dilakukan berikutnya?

### Jawaban Refleksi Akhir
1. **Apa hasil utama yang Anda amati?**
   Bahwa algoritma klasifikasi tidak bisa dilepas begitu saja. Keduanya (k-NN dan Tree) membutuhkan pembatasan *hyperparameter* (nilai `k` dan `max_depth`) yang tepat agar performa model seimbang dan tidak anjlok saat menebak data baru.

2. **Mengapa hasil tersebut terjadi menurut konsep di Main Deck?**
   Karena model yang diberi kebebasan mutlak (*unrestricted tree* atau *k=1*) cenderung mempelajari dan menghafal anomali (*noise*) yang spesifik hanya ada pada data *train*, alih-alih menangkap pola umum. Hal ini sesuai dengan peringatan *Common Error 3* di modul.

3. **Apa limitation eksperimen ini?**
   Eksperimen ini hanya dilakukan pada dataset "Iris" yang berukuran sangat kecil (150 sampel), terstruktur rapi, dan bersih dari *noise*. Tingginya skor akurasi di sini tidak menggambarkan kerasnya tantangan menangani dataset kotor di dunia nyata.

4. **Eksperimen apa yang layak dilakukan berikutnya?**
   Menggunakan *Cross-Validation* alih-alih hanya satu kali *split* agar evaluasi metrik lebih objektif dan stabil. Selain itu, menguji kedua model ini pada dataset yang memiliki *outlier* (data pencilan) yang ekstrem.